# MongoDB Atlas, Basic MQL, and Document Modeling

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/CST4714_OER_Rebuild/notebooks/04_atlas_mql_modeling.ipynb)

This notebook introduces MongoDB through direct, visible operations. You will
insert a small synthetic fixture, query nested fields and arrays, update one test
document, interpret write evidence, and compare embedded and referenced shapes.

The default path uses an in-memory MongoDB-compatible library. Set `USE_ATLAS` to
`True` only in class when your free Atlas cluster, database user, and temporary
network rule are ready.

## Atlas Connection Checklist

1. Use a Free cluster; no paid tier is required.
2. Create a database user distinct from your Atlas website account.
3. Add only the temporary network access needed for class and narrow or remove it
   afterward.
4. Copy the current driver connection URI.
5. Enter the URI only through `getpass` below.

Do not set `tlsInsecure=True`. A TLS error is a signal to check the current driver,
URI, DNS, network rule, system time, and certificate path.

In [1]:
%pip -q install pymongo mongomock

Note: you may need to restart the kernel to use updated packages.


In [2]:
from datetime import datetime, timezone
from getpass import getpass

import mongomock
from pymongo import MongoClient

USE_ATLAS = False  # Change to True only when your Atlas setup is ready.
DATABASE_NAME = "cst4714_metro_support_practice"  # Make unique in a shared project.

if USE_ATLAS:
    mongodb_uri = getpass("Paste the temporary Atlas URI: ")
    client = MongoClient(mongodb_uri, serverSelectionTimeoutMS=10000)
    print("Atlas ping:", client.admin.command("ping"))
else:
    client = mongomock.MongoClient()
    print("Using the offline in-memory MongoDB-compatible path.")

database = client[DATABASE_NAME]
tickets = database["tickets"]

Using the offline in-memory MongoDB-compatible path.


## 1. Load a Small, Reproducible Fixture

Every course document carries `course_fixture: "cst4714"`. Cleanup filters on that
marker instead of dropping an entire database. The fixture uses BSON dates through
Python `datetime` values, nested requester documents, tag arrays, and embedded
event arrays.

In [3]:
tickets.delete_many({"course_fixture": "cst4714"})

fixture = [
    {
        "ticket_id": 1001,
        "category": "streetlight",
        "priority": "high",
        "status": "open",
        "subject": "Streetlight dark near bus stop",
        "requester": {"user_id": 101, "display_name": "Maya Chen"},
        "assignee_id": 201,
        "opened_at": datetime(2026, 2, 1, 23, 10, tzinfo=timezone.utc),
        "tags": ["lighting", "safety"],
        "events": [
            {"event_id": 5001, "type": "created", "actor_role": "resident",
             "at": datetime(2026, 2, 1, 23, 10, tzinfo=timezone.utc)},
            {"event_id": 5002, "type": "assigned", "actor_role": "agent",
             "at": datetime(2026, 2, 2, 14, 5, tzinfo=timezone.utc)},
        ],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1002,
        "category": "sanitation",
        "priority": "medium",
        "status": "in_progress",
        "subject": "Missed recycling pickup",
        "requester": {"user_id": 102, "display_name": "Luis Rivera"},
        "assignee_id": 202,
        "opened_at": datetime(2026, 2, 2, 15, 45, tzinfo=timezone.utc),
        "tags": ["recycling"],
        "events": [
            {"event_id": 5003, "type": "created", "actor_role": "resident",
             "at": datetime(2026, 2, 2, 15, 45, tzinfo=timezone.utc)},
            {"event_id": 5004, "type": "status_changed", "actor_role": "agent",
             "at": datetime(2026, 2, 3, 13, 30, tzinfo=timezone.utc)},
        ],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1003,
        "category": "water",
        "priority": "urgent",
        "status": "resolved",
        "subject": "Low water pressure",
        "requester": {"user_id": 103, "display_name": "Amina Yusuf"},
        "assignee_id": 201,
        "opened_at": datetime(2026, 2, 3, 12, 5, tzinfo=timezone.utc),
        "tags": ["water", "building"],
        "events": [
            {"event_id": 5005, "type": "created", "actor_role": "resident",
             "at": datetime(2026, 2, 3, 12, 5, tzinfo=timezone.utc)},
            {"event_id": 5006, "type": "status_changed", "actor_role": "agent",
             "at": datetime(2026, 2, 3, 14, 25, tzinfo=timezone.utc)},
            {"event_id": 5007, "type": "status_changed", "actor_role": "agent",
             "at": datetime(2026, 2, 3, 19, 40, tzinfo=timezone.utc)},
        ],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1004,
        "category": "parks",
        "priority": "low",
        "status": "new",
        "subject": "Broken bench slat",
        "requester": {"user_id": 104, "display_name": "Jordan Bell"},
        "assignee_id": None,
        "opened_at": datetime(2026, 2, 4, 17, 20, tzinfo=timezone.utc),
        "tags": ["parks"],
        "events": [
            {"event_id": 5008, "type": "created", "actor_role": "resident",
             "at": datetime(2026, 2, 4, 17, 20, tzinfo=timezone.utc)}
        ],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1005,
        "category": "sanitation",
        "priority": "high",
        "status": "resolved",
        "subject": "Overflowing corner bin",
        "requester": {"user_id": 101, "display_name": "Maya Chen"},
        "assignee_id": 202,
        "opened_at": datetime(2026, 2, 5, 14, 0, tzinfo=timezone.utc),
        "tags": ["sanitation", "safety"],
        "events": [],
        "course_fixture": "cst4714",
    },
    {
        "ticket_id": 1006,
        "category": "streetlight",
        "priority": "medium",
        "status": "in_progress",
        "subject": "Flickering lamp outside library",
        "requester": {"user_id": 102, "display_name": "Luis Rivera"},
        "assignee_id": 201,
        "opened_at": datetime(2026, 2, 6, 1, 30, tzinfo=timezone.utc),
        "tags": ["lighting", "library"],
        "events": [],
        "course_fixture": "cst4714",
    },
]

insert_result = tickets.insert_many(fixture)
print("Inserted documents:", len(insert_result.inserted_ids))
print("Verified fixture count:", tickets.count_documents({"course_fixture": "cst4714"}))

Inserted documents: 6
Verified fixture count: 6


## 2. Filter, Project, and Sort

The result grain is one document per matching ticket. The projection excludes
`_id` and returns only fields needed for the question.

In [4]:
active_high_priority = tickets.find(
    {
        "course_fixture": "cst4714",
        "status": {"$in": ["new", "open", "in_progress"]},
        "priority": {"$in": ["high", "urgent"]},
    },
    {"_id": 0, "ticket_id": 1, "priority": 1, "status": 1, "subject": 1, "opened_at": 1},
).sort("opened_at", -1)

for document in active_high_priority:
    print(document)

{'ticket_id': 1001, 'priority': 'high', 'status': 'open', 'subject': 'Streetlight dark near bus stop', 'opened_at': datetime.datetime(2026, 2, 1, 23, 10)}


### Your Turn

Modify the next filter to choose a different status set or category, and modify the
projection to add exactly one useful field. State the expected result grain before
running it.

In [5]:
# Grain: one document per matching ticket.
for document in tickets.find(
    {"course_fixture": "cst4714", "category": "streetlight"},
    {"_id": 0, "ticket_id": 1, "category": 1, "status": 1, "subject": 1},
).sort("ticket_id", 1):
    print(document)

{'ticket_id': 1001, 'category': 'streetlight', 'status': 'open', 'subject': 'Streetlight dark near bus stop'}
{'ticket_id': 1006, 'category': 'streetlight', 'status': 'in_progress', 'subject': 'Flickering lamp outside library'}


## 3. Query a Nested Field and an Array

Dot notation reaches `requester.user_id`. Equality against an array field matches
when the array contains that value.

In [6]:
print("Tickets requested by user 101:")
for document in tickets.find(
    {"course_fixture": "cst4714", "requester.user_id": 101},
    {"_id": 0, "ticket_id": 1, "requester.display_name": 1, "status": 1},
):
    print(document)

print("\nTickets tagged safety:")
for document in tickets.find(
    {"course_fixture": "cst4714", "tags": "safety"},
    {"_id": 0, "ticket_id": 1, "tags": 1},
):
    print(document)

Tickets requested by user 101:
{'ticket_id': 1001, 'status': 'open', 'requester': {'display_name': 'Maya Chen'}}
{'ticket_id': 1005, 'status': 'resolved', 'requester': {'display_name': 'Maya Chen'}}

Tickets tagged safety:
{'ticket_id': 1001, 'tags': ['lighting', 'safety']}
{'ticket_id': 1005, 'tags': ['sanitation', 'safety']}


## 4. `$elemMatch` Requires Conditions on the Same Array Element

The question asks for one event whose type is `status_changed` **and** whose actor
role is `agent`. `$elemMatch` prevents one array element from satisfying the type
while a different element satisfies the actor condition.

In [7]:
for document in tickets.find(
    {
        "course_fixture": "cst4714",
        "events": {
            "$elemMatch": {"type": "status_changed", "actor_role": "agent"}
        },
    },
    {"_id": 0, "ticket_id": 1, "events": 1},
):
    print(document)

{'ticket_id': 1002, 'events': [{'event_id': 5003, 'type': 'created', 'actor_role': 'resident', 'at': datetime.datetime(2026, 2, 2, 15, 45)}, {'event_id': 5004, 'type': 'status_changed', 'actor_role': 'agent', 'at': datetime.datetime(2026, 2, 3, 13, 30)}]}
{'ticket_id': 1003, 'events': [{'event_id': 5005, 'type': 'created', 'actor_role': 'resident', 'at': datetime.datetime(2026, 2, 3, 12, 5)}, {'event_id': 5006, 'type': 'status_changed', 'actor_role': 'agent', 'at': datetime.datetime(2026, 2, 3, 14, 25)}, {'event_id': 5007, 'type': 'status_changed', 'actor_role': 'agent', 'at': datetime.datetime(2026, 2, 3, 19, 40)}]}


## 5. Verify Matched and Modified Counts

The lab inserts one clearly marked test document. The first `$set` changes it. The
second identical `$set` still matches the document but has no new value to write.

In [8]:
test_document = {
    "ticket_id": 1099,
    "category": "parks",
    "priority": "low",
    "status": "new",
    "subject": "Disposable MQL test",
    "requester": {"user_id": 101, "display_name": "Maya Chen"},
    "opened_at": datetime.now(timezone.utc),
    "events": [],
    "test_record": True,
    "course_fixture": "cst4714",
}
tickets.insert_one(test_document)

first_update = tickets.update_one(
    {"ticket_id": 1099, "test_record": True, "status": "new"},
    {"$set": {"status": "in_progress", "assignee_id": 202}},
)
print("First update matched/modified:", first_update.matched_count, first_update.modified_count)

second_update = tickets.update_one(
    {"ticket_id": 1099, "test_record": True, "status": "in_progress"},
    {"$set": {"status": "in_progress", "assignee_id": 202}},
)
print("Repeated update matched/modified:", second_update.matched_count, second_update.modified_count)

First update matched/modified: 1 1
Repeated update matched/modified: 1 0


## 6. Append One Event and Read Back the Final Document

`$push` appends to the array. In a production event flow, use a stable event ID and
idempotency rule so a retry cannot append the same event twice.

In [9]:
event_update = tickets.update_one(
    {"ticket_id": 1099, "test_record": True},
    {
        "$push": {
            "events": {
                "event_id": 5999,
                "type": "status_changed",
                "actor_role": "agent",
                "at": datetime.now(timezone.utc),
            }
        }
    },
)
print("Event append matched/modified:", event_update.matched_count, event_update.modified_count)
print(tickets.find_one({"ticket_id": 1099}, {"_id": 0}))

Event append matched/modified: 1 1
{'ticket_id': 1099, 'category': 'parks', 'priority': 'low', 'status': 'in_progress', 'subject': 'Disposable MQL test', 'requester': {'user_id': 101, 'display_name': 'Maya Chen'}, 'opened_at': datetime.datetime(2026, 7, 13, 6, 29, 48, 825000), 'events': [{'event_id': 5999, 'type': 'status_changed', 'actor_role': 'agent', 'at': datetime.datetime(2026, 7, 13, 6, 29, 48, 829000)}], 'test_record': True, 'course_fixture': 'cst4714', 'assignee_id': 202}


## 7. Delete Only the Disposable Record

The predicate includes both the identifier and the safety marker. The final query
proves cleanup.

In [10]:
delete_result = tickets.delete_one({"ticket_id": 1099, "test_record": True})
print("Deleted count:", delete_result.deleted_count)
print("Remaining test record:", tickets.find_one({"ticket_id": 1099, "test_record": True}))

Deleted count: 1
Remaining test record: None


## 8. Compare Two Models

These are design sketches, not additional database writes.

In [11]:
embedded_ticket = {
    "ticket_id": 1001,
    "requester": {"user_id": 101, "display_name": "Maya Chen"},
    "events": [{"event_id": 5001, "type": "created"}],
}

referenced_ticket = {
    "ticket_id": 1001,
    "requester_id": 101,
    "event_ids": [5001],
}

print("Embedded sketch:", embedded_ticket)
print("Referenced sketch:", referenced_ticket)

Embedded sketch: {'ticket_id': 1001, 'requester': {'user_id': 101, 'display_name': 'Maya Chen'}, 'events': [{'event_id': 5001, 'type': 'created'}]}
Referenced sketch: {'ticket_id': 1001, 'requester_id': 101, 'event_ids': [5001]}


## Evidence Record: Complete Before Submission

**Modified filter and projection:** [what you changed and what one result document
represented]

**Nested/array reasoning:** [which query used dot notation and why `$elemMatch` did
or did not matter]

**Write evidence:** [interpret the first and repeated matched/modified counts]

**Delete safety:** [why the exact predicate could not delete the fixture broadly]

**Model choice:** [embed or reference events for one stated access pattern, with
growth and duplication tradeoff]

**Atlas control:** [if used, name one database-user or network control and how you
narrowed or removed it]

**Credential check:** I confirm no URI or password appears in notebook source or
output. [replace with yes]

**License:** prose CC BY-NC-SA 4.0; code MIT; synthetic data CC0.

In [12]:
client.close()
print("Client closed. Baseline fixture retained for the next course lab when Atlas was used.")

Client closed. Baseline fixture retained for the next course lab when Atlas was used.
